# Hafta 8 · Oracle Kavramı ve İlk Kuantum Algoritmaları
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta ilk gerçek kuantum algoritmalarını yazıyoruz: **Deutsch**, **Deutsch-Jozsa (DJ)** ve **Bernstein-Vazirani (BV)**. Hepsi aynı kalıbı kullanır: *H → oracle (1 kez) → H → ölç*. Lab'ın merkezinde bir **oracle fabrikası** var: doğruluk tablosundan, gizli stringten ya da rastgele dengeli fonksiyondan oracle devresi üreten fonksiyonlar. Her oracle çağrısını bir **sorgu sayacı** ile sayıp klasik algoritmalarla karşılaştıracağız.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Oracle = kara kutu fonksiyon, klasik sorgu sayacı | 5 dk |
| B | Bit oracle, faz oracle ve faz geri tepmesi (Statevector ile adım adım) | 8 dk |
| C | Walsh-Hadamard dönüşümü H⊗n | 5 dk |
| D | Deutsch algoritması (1 bit, 4 fonksiyon) | 6 dk |
| E | **Oracle fabrikası** | 8 dk |
| F | Deutsch-Jozsa + klasik karşılaştırma (deterministik ve olasılıksal) | 8 dk |
| G | Bernstein-Vazirani (6–8 bit) | 5 dk |
| H | Simon'a kısa bakış | 2 dk |
| I | Alıştırmalar (8 adet) | ödev |

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
rng = np.random.default_rng(2026)
aer = AerSimulator(seed_simulator=2026)

def run_counts(qc, shots=1024):
    """Devreyi Aer simülatöründe çalıştırıp sayımları döndürür."""
    return aer.run(transpile(qc, aer), shots=shots).result().get_counts()

def show_amps(sv, n, title="", tol=1e-9):
    """Durum vektörünü okunaklı tablo olarak yazdırır (sıfır olmayan genlikler)."""
    print(title)
    for i, a in enumerate(np.asarray(sv)):
        if abs(a) > tol:
            print(f"  |{format(i, f'0{n}b')}⟩ : {a.real:+.3f}" + (f"{a.imag:+.3f}i" if abs(a.imag) > tol else ""))

def bar_counts(counts, title=""):
    keys = sorted(counts); vals = [counts[k] for k in keys]
    plt.figure(figsize=(max(4, 0.55*len(keys)), 2.8))
    plt.bar(keys, vals, color=BLUE, width=0.6); plt.title(title, color=NAVY); plt.xticks(rotation=60); plt.show()
print("hazır")

---
## A · Oracle = kara kutu fonksiyon
**Oracle**, içini göremediğimiz ama **sorabildiğimiz** bir fonksiyondur. Tıpkı kaynak kodunu bilmediğimiz bir API gibi: `GET /f?x=1011 → 0`.
Bu hafta **maliyet = sorgu (oracle çağrısı) sayısı**. Buna *sorgu karmaşıklığı* denir.

Deutsch-Jozsa problemi: f: {0,1}ⁿ → {0,1} ya **sabittir** (hep 0 ya da hep 1) ya da **dengelidir** (girdilerin tam yarısında 0, yarısında 1). Hangisi?
Klasik ve deterministik bir algoritma en kötü durumda **2ⁿ⁻¹ + 1** sorgu yapmak zorundadır: ilk yarının hepsi aynı çıkabilir.

In [ ]:
class CountingOracle:
    """Klasik oracle: f'yi sarmalar ve her çağrıyı sayar (API çağrı sayacı gibi)."""
    def __init__(self, table):
        self.table = list(table); self.n = int(np.log2(len(table))); self.queries = 0
    def __call__(self, x):
        self.queries += 1
        return self.table[x]

def classical_dj(f):
    """Deterministik klasik DJ: farklı iki değer görene kadar ya da 2^(n-1)+1 sorguya kadar sor."""
    first = f(0)
    for x in range(1, 2**(f.n - 1) + 1):
        if f(x) != first:
            return "dengeli"
    return "sabit"

n = 4
sabit = CountingOracle([1] * 2**n)
kotu_dengeli = CountingOracle([0] * 2**(n-1) + [1] * 2**(n-1))   # ilk yarı hep 0 -> en kötü durum
print("sabit        :", classical_dj(sabit), " sorgu =", sabit.queries)
print("kötü dengeli :", classical_dj(kotu_dengeli), " sorgu =", kotu_dengeli.queries)
print("en kötü durum formülü 2^(n-1)+1 =", 2**(n-1) + 1)

---
## B · Bit oracle, faz oracle ve faz geri tepmesi
Kuantum devreleri **tersinir** (üniter) olmak zorunda. `|x⟩ → |f(x)⟩` genelde tersinir değildir (iki girdi aynı çıktıya gidebilir). Çözüm: fazladan bir **yardımcı (hedef) kübit** ekleyip sonucu ona XOR'lamak:

$$U_f\,|x\rangle|y\rangle = |x\rangle\,|y \oplus f(x)\rangle \qquad \text{(bit oracle)}$$

Hedef kübit **|−⟩** durumunda ise ilginç bir şey olur: hedef değişmez, onun yerine girdiye bir **işaret** gelir:

$$U_f\,|x\rangle|-\rangle = (-1)^{f(x)}\,|x\rangle|-\rangle \qquad \text{(faz geri tepmesi / phase kickback)}$$

Neden? |−⟩ = (|0⟩ − |1⟩)/√2. f(x)=1 ise X uygulanır ve |−⟩ → (|1⟩ − |0⟩)/√2 = −|−⟩. Eksi işaret "ortak" olduğu için girdi kübitine yazılabilir. Aşağıda genliklere bakarak doğrulayalım.

In [ ]:
# 1 girdi kübiti (q0 = x) + 1 hedef kübit (q1 = y).  f(x) = x  ->  oracle = CNOT(0, 1)
for x in [0, 1]:
    qc = QuantumCircuit(2)
    if x: qc.x(0)
    qc.x(1); qc.h(1)                       # hedef = |−⟩
    before = Statevector(qc)
    qc.cx(0, 1)                             # U_f
    after = Statevector(qc)
    show_amps(before.data, 2, f"x = {x}, U_f ÖNCESİ (q1 q0):")
    show_amps(after.data, 2,  f"x = {x}, U_f SONRASI:")
    print(f"  -> sonrası = {(-1)**x:+d} × öncesi ? ", np.allclose(after.data, (-1)**x * before.data), "\n")

In [ ]:
# Süperpozisyonda: girdi |+⟩ iken CNOT, girdiyi |−⟩'ye çevirir (hedef değişmeden!)
qc = QuantumCircuit(2); qc.h(0); qc.x(1); qc.h(1)
show_amps(Statevector(qc).data, 2, "Önce  (q0=|+⟩, q1=|−⟩):")
qc.cx(0, 1)
show_amps(Statevector(qc).data, 2, "Sonra (q0=|−⟩, q1=|−⟩):")
ref = Statevector.from_label("--")
print("Sonuç |−⟩|−⟩ mi?", Statevector(qc).equiv(ref))
qc.draw("mpl")

**Bit oracle mu, faz oracle mu?** İkisi aynı fonksiyonu kodlar. Faz oracle'ı doğrudan köşegen matris olarak da yazabiliriz: `diag((-1)^f(0), ..., (-1)^f(2ⁿ-1))`. f(x) = x₀ ⊕ x₁ için faz oracle'ı yalnızca iki Z kapısıdır.

In [ ]:
tt = [0, 1, 1, 0]                                  # f(x) = x0 XOR x1
D = np.diag([(-1)**v for v in tt])
qz = QuantumCircuit(2); qz.z(0); qz.z(1)
print("Faz oracle matrisi köşegeni:", np.diag(D))
print("Z⊗Z ile aynı mı?", np.allclose(Operator(qz).data, D))

---
## C · Walsh-Hadamard dönüşümü H⊗n
Her kübite H uygulamak (H⊗n):
- |0…0⟩'dan **eşit süperpozisyon** üretir: tüm 2ⁿ durum 1/√2ⁿ genlikli.
- Genel olarak: $H^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}}\sum_z (-1)^{x\cdot z}|z\rangle$, burada x·z = (x AND z)'deki 1'lerin sayısı (mod 2).
Kodda: `(-1) ** bin(x & z).count("1")`.

In [ ]:
def dot2(x, z):
    """Bit dizilerinin mod 2 iç çarpımı: x·z = popcount(x & z) mod 2."""
    return bin(x & z).count("1") % 2

n = 3
qc = QuantumCircuit(n); qc.h(range(n))
HN = Operator(qc).data.real * np.sqrt(2**n)       # ±1 matrisi
formula = np.array([[(-1)**dot2(x, z) for x in range(2**n)] for z in range(2**n)])
print("H⊗3 işaret matrisi = (−1)^(x·z) ?", np.allclose(HN, formula))

plt.figure(figsize=(4.2, 3.8))
plt.imshow(formula, cmap="coolwarm_r"); plt.title("H⊗3 işaret deseni (mavi +, kırmızı −)", color=NAVY, fontsize=10)
L = [format(i, "03b") for i in range(8)]; plt.xticks(range(8), L, rotation=60); plt.yticks(range(8), L)
plt.xlabel("girdi x"); plt.ylabel("çıktı z"); plt.show()

---
## D · Deutsch algoritması (n = 1)
1 bitlik 4 fonksiyon var: f₀ = 0, f₁ = 1 (sabit), f₂ = x, f₃ = NOT x (dengeli). Klasikte cevap için **2 sorgu** gerekir (f(0) ve f(1)). Deutsch devresi **1 sorgu** ile karar verir:

`q1: X → H → [U_f] ;   q0: H → [U_f] → H → ölç`   →   ölçüm 0 ise sabit, 1 ise dengeli.

In [ ]:
def deutsch_oracle(k):
    """k = 0: f=0, 1: f=1, 2: f=x, 3: f=NOT x.  q0 = x, q1 = y."""
    o = QuantumCircuit(2, name=f"f{k}")
    if k == 1: o.x(1)
    if k == 2: o.cx(0, 1)
    if k == 3: o.cx(0, 1); o.x(1)
    return o

def deutsch_circuit(oracle):
    qc = QuantumCircuit(2, 1)
    qc.x(1); qc.h([0, 1]); qc.barrier()
    qc.append(oracle.to_gate(label="U_f"), [0, 1]); qc.barrier()
    qc.h(0); qc.measure(0, 0)
    return qc

for k, name in enumerate(["f=0", "f=1", "f=x", "f=NOT x"]):
    c = run_counts(deutsch_circuit(deutsch_oracle(k)), shots=200)
    print(f"{name:8s} -> {c}  ->  {'SABİT' if '0' in c else 'DENGELİ'}")
deutsch_circuit(deutsch_oracle(2)).draw("mpl")

In [ ]:
# Adım adım genlikler (f = x): Statevector ile her adımdan sonra durum
steps = [("X(q1)", lambda q: q.x(1)), ("H⊗H", lambda q: q.h([0, 1])),
         ("U_f", lambda q: q.compose(deutsch_oracle(2), inplace=True)), ("H(q0)", lambda q: q.h(0))]
qc = QuantumCircuit(2)
for name, op in steps:
    op(qc); show_amps(Statevector(qc).data, 2, f"{name} sonrası:")

---
## E · Oracle fabrikası
Gerçek dünyada oracle "birinin verdiği" bir devredir. Laboratuvarda ise onları kendimiz üretiyoruz. Üç fabrika fonksiyonu yazacağız (hepsi n+1 kübitli devre döndürür; q0…q(n−1) girdi, qn hedef):

| Fonksiyon | Girdi | Yöntem |
|---|---|---|
| `oracle_from_truth_table(tt)` | doğruluk tablosu (2ⁿ uzunluk) | f(x)=1 olan her x için **x'e koşullu çoklu kontrollü X** (`mcx` + `ctrl_state`) |
| `bv_oracle(s)` | gizli string, ör. `'1011'` | s'nin 1 olan her biti için **CNOT(i → hedef)** |
| `random_balanced_oracle(n, seed)` | n ve tohum | rastgele **XOR maskeli parite**: X(maske) → CNOT'lar → X(maske) [+ çıkış X] |

Ayrıca `truth_table_of(oracle, n)` ile herhangi bir oracle devresinin doğruluk tablosunu **geri okuyup** test edebileceğiz (birim test!).

In [ ]:
def oracle_from_truth_table(tt):
    """Doğruluk tablosundan bit oracle: |x⟩|y⟩ -> |x⟩|y ⊕ f(x)⟩."""
    n = int(np.log2(len(tt))); assert len(tt) == 2**n
    qc = QuantumCircuit(n + 1, name="U_f")
    for x, fx in enumerate(tt):
        if fx:
            qc.mcx(list(range(n)), n, ctrl_state=x)     # tüm girdi bitleri x'e eşitse hedefi çevir
    return qc

def bv_oracle(s):
    """f(x) = s·x mod 2.  s[-1] <-> q0 (Qiskit sırası)."""
    n = len(s); qc = QuantumCircuit(n + 1, name="U_f")
    for i, bit in enumerate(reversed(s)):
        if bit == "1":
            qc.cx(i, n)
    return qc

def random_balanced_oracle(n, seed=None):
    """Rastgele dengeli oracle: f(x) = (s·(x⊕m)) ⊕ c,  s ≠ 0.  Parite fonksiyonu tam dengelidir."""
    r = np.random.default_rng(seed)
    s = int(r.integers(1, 2**n)); m = int(r.integers(0, 2**n)); c = int(r.integers(0, 2))
    qc = QuantumCircuit(n + 1, name="U_f")
    mask = [i for i in range(n) if (m >> i) & 1]
    if mask: qc.x(mask)
    for i in range(n):
        if (s >> i) & 1: qc.cx(i, n)
    if mask: qc.x(mask)
    if c: qc.x(n)
    return qc

def constant_oracle(n, value):
    qc = QuantumCircuit(n + 1, name="U_f")
    if value: qc.x(n)
    return qc

def truth_table_of(oracle, n):
    """Oracle'ı her klasik girdi x ile çalıştırıp hedef bitten f(x)'i okur (klasik sorgu, 2^n kez)."""
    tt = []
    for x in range(2**n):
        qc = QuantumCircuit(n + 1)
        for i in range(n):
            if (x >> i) & 1: qc.x(i)
        qc.compose(oracle, inplace=True)
        probs = Statevector(qc).probabilities([n])       # yalnız hedef kübitin olasılıkları
        tt.append(int(round(probs[1])))
    return tt

tt = [0, 1, 1, 0]
print("tablo -> oracle -> tablo:", truth_table_of(oracle_from_truth_table(tt), 2))
print("BV s='101' tablosu      :", truth_table_of(bv_oracle("101"), 3))
rb = random_balanced_oracle(4, seed=7)
t = truth_table_of(rb, 4); print("rastgele dengeli (n=4)  :", t, " -> 1 sayısı =", sum(t))
oracle_from_truth_table(tt).draw("mpl")

In [ ]:
rb.draw("mpl")   # XOR maskeli dengeli oracle örneği

---
## F · Deutsch-Jozsa
Devre: girdi kübitlerine H, hedefe X+H (|−⟩), **oracle bir kez**, girdilere tekrar H, ölç.
- Sonuç `00…0` → **sabit**,  diğer her sonuç → **dengeli** (olasılık 1 ile, hatasız).

Neden? Son H⊗n'den sonra |0…0⟩ genliği = (−1)^f(x) işaretlerinin **ortalaması**. Sabitte ortalama ±1, dengelide tam 0.

In [ ]:
def dj_circuit(oracle, n):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n); qc.h(range(n + 1)); qc.barrier()
    qc.append(oracle.to_gate(label="U_f"), range(n + 1)); qc.barrier()
    qc.h(range(n)); qc.measure(range(n), range(n))
    return qc

def oracle_calls(qc):
    """Devrede oracle kapısının kaç kez kullanıldığını sayar (kuantum sorgu sayacı)."""
    return sum(1 for ins in qc.data if ins.operation.name == "U_f")

n = 3
tests = {"sabit 0": constant_oracle(n, 0), "sabit 1": constant_oracle(n, 1),
         "dengeli (tablo)": oracle_from_truth_table([0, 1, 1, 0, 1, 0, 0, 1]),
         "dengeli (rastgele)": random_balanced_oracle(n, seed=3)}
for name, o in tests.items():
    qc = dj_circuit(o, n); c = run_counts(qc, 256)
    print(f"{name:20s} sayımlar={c}   oracle çağrısı={oracle_calls(qc)}")
dj_circuit(tests["dengeli (tablo)"], n).draw("mpl")

In [ ]:
# Adım adım Statevector izleme (n = 2, dengeli f = x0 ⊕ x1). Hedef kübit q2.
n = 2; o = oracle_from_truth_table([0, 1, 1, 0])
qc = QuantumCircuit(n + 1)
qc.x(n); qc.h(range(n + 1)); show_amps(Statevector(qc).data, n + 1, "H'ler sonrası (q2 q1 q0):")
qc.compose(o, inplace=True);   show_amps(Statevector(qc).data, n + 1, "Oracle sonrası (girdi fazları değişti):")
qc.h(range(n));                show_amps(Statevector(qc).data, n + 1, "Son H'ler sonrası:")
pr = Statevector(qc).probabilities(qargs=[0, 1])
print("Girdi kübitlerinin olasılıkları:", {format(i, "02b"): round(float(p), 3) for i, p in enumerate(pr)})

### Klasik ile karşılaştırma: deterministik ve olasılıksal
Dürüst olalım: **olasılıksal** bir klasik algoritma (k rastgele girdi sor; hepsi aynıysa "sabit" de) dengeli bir fonksiyonu ancak 2^(1−k) olasılıkla yanlış sınıflandırır. k = 10 ile hata < %0.2 — ve bu **n'den bağımsızdır**. Yani DJ'nin üstel avantajı yalnızca "%100 kesinlik" istendiğinde geçerlidir.

In [ ]:
def classical_random_dj(f, k, seed=None):
    r = np.random.default_rng(seed)
    xs = r.integers(0, 2**f.n, size=k)
    vals = {f(int(x)) for x in xs}
    return "sabit" if len(vals) == 1 else "dengeli"

n, k, trials = 10, 10, 2000
wrong = 0
for t in range(trials):
    tt = np.zeros(2**n, int); tt[rng.permutation(2**n)[:2**(n-1)]] = 1     # rastgele dengeli
    f = CountingOracle(tt)
    wrong += classical_random_dj(f, k, seed=t) == "sabit"
print(f"n={n}, k={k}: hata oranı = {wrong/trials:.4f}   (teorik üst sınır 2^(1-k) = {2**(1-k):.4f})")

ns = np.arange(1, 16)
plt.figure(figsize=(6, 3.2))
plt.semilogy(ns, 2.0**(ns-1) + 1, "o-", color=NAVY, label="klasik deterministik")
plt.semilogy(ns, np.full(len(ns), k), "s--", color=GRAY, label=f"klasik olasılıksal (k={k})")
plt.semilogy(ns, np.ones(len(ns)), "D-", color=BLUE, label="Deutsch-Jozsa")
plt.xlabel("n"); plt.ylabel("sorgu"); plt.legend(frameon=False); plt.title("Sorgu sayısı", color=NAVY); plt.show()

---
## G · Bernstein-Vazirani
Gizli bir bit maskesi s var ve oracle f(x) = s·x mod 2 döndürüyor (x'in s ile seçilen bitlerinin paritesi).
- Klasik: her sorgu **1 bit** bilgi verir → x = 00…01, 00…10, … ile **n sorgu**.
- Kuantum: DJ ile **aynı devre**, tek sorgu; ölçüm sonucu doğrudan **s**'dir.

Benzetme: Bir API'nin hangi özellik bayraklarını (feature flags) dikkate aldığını keşfetmek. Klasikte her bayrağı tek tek açıp denersiniz; BV hepsini tek istekte söyler.

In [ ]:
def bv_circuit(oracle, n):
    return dj_circuit(oracle, n)            # devre aynı!

for s in ["101101", "11100101", "00000001"]:
    n = len(s); qc = bv_circuit(bv_oracle(s), n)
    c = run_counts(qc, 512)
    print(f"s = {s}  ->  ölçüm {c}  oracle çağrısı = {oracle_calls(qc)}")
bv_circuit(bv_oracle("101101"), 6).draw("mpl", fold=-1)

In [ ]:
# Klasik BV: n sorgu (sayaçla)
s = "11100101"; n = len(s)
f = CountingOracle([bin(x & int(s, 2)).count("1") % 2 for x in range(2**n)])
bits = [f(1 << i) for i in range(n)]                   # e_i = sadece i. biti 1
found = "".join(str(b) for b in reversed(bits))
print("klasik bulunan s =", found, " sorgu =", f.queries)

---
## H · Simon'a kısa bakış
Simon probleminde f 2'ye-1'dir: f(x) = f(x ⊕ s). Kuantum devresi her çalıştırmada s·z = 0 koşulunu sağlayan rastgele bir z verir; yaklaşık n farklı z toplanıp lineer denklem sistemi (mod 2) çözülünce s bulunur. Klasikte olasılıksal algoritmalar bile ~2^(n/2) sorgu gerektirir → **üstel ayrım**. Aşağıda n = 2, s = 11 için ölçülen z'lerin hep s·z = 0 sağladığını görüyoruz.

In [ ]:
qc = QuantumCircuit(4, 2); qc.h([0, 1])
qc.cx(0, 2); qc.cx(0, 3); qc.cx(1, 2); qc.cx(1, 3)      # f(x) = (x0⊕x1, x0⊕x1)  -> s = 11
qc.h([0, 1]); qc.measure([0, 1], [0, 1])
c = run_counts(qc, 512); print(c)
print("Tüm z'ler için s·z = 0 ?", all(dot2(int(z, 2), 0b11) == 0 for z in c))

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Faz oracle'ını NumPy ile uygula
`apply_phase_oracle(state, tt)`: durum vektörünün x. genliğini (−1)^tt[x] ile çarpsın.

In [ ]:
def apply_phase_oracle(state, tt):
    # TODO
    pass

psi = np.ones(4) / 2
out = apply_phase_oracle(psi, [0, 1, 1, 0])
assert np.allclose(out, [0.5, -0.5, -0.5, 0.5])
# Bit oracle + |−⟩ hedef ile aynı sonucu vermeli (faz geri tepmesi):
qc = QuantumCircuit(3); qc.h([0, 1]); qc.x(2); qc.h(2); qc.compose(oracle_from_truth_table([0, 1, 1, 0]), inplace=True)
assert np.allclose(Statevector(qc).data, np.kron([1, -1] / np.sqrt(2), out))
print("Alıştırma 1 ✓")

### Alıştırma 2 · Walsh-Hadamard matrisi
`walsh_hadamard(n)`: (1/√2ⁿ)·(−1)^(x·z) elemanlı 2ⁿ×2ⁿ matrisi **döngüyle/formülle** kursun (np.kron kullanmadan). Qiskit'in H⊗n operatörüyle karşılaştırın.

In [ ]:
def walsh_hadamard(n):
    # TODO
    pass

for n in [1, 2, 3, 4]:
    qc = QuantumCircuit(n); qc.h(range(n))
    assert np.allclose(walsh_hadamard(n), Operator(qc).data)
print("Alıştırma 2 ✓")

### Alıştırma 3 · Deutsch kararı
`deutsch_decide(oracle)`: `deutsch_circuit` ile **tek shot** çalıştırıp `"sabit"` ya da `"dengeli"` döndürsün.

In [ ]:
def deutsch_decide(oracle):
    # TODO
    pass

assert [deutsch_decide(deutsch_oracle(k)) for k in range(4)] == ["sabit", "sabit", "dengeli", "dengeli"]
print("Alıştırma 3 ✓")

### Alıştırma 4 · Deutsch-Jozsa kararı (n = 4)
`dj_decide(oracle, n)`: DJ devresini **tek shot** çalıştırsın; sonuç `'0000'` ise `"sabit"`, değilse `"dengeli"`. 10 rastgele dengeli ve 2 sabit oracle ile test edilir.

In [ ]:
def dj_decide(oracle, n):
    # TODO
    pass

n = 4
assert dj_decide(constant_oracle(n, 0), n) == "sabit" and dj_decide(constant_oracle(n, 1), n) == "sabit"
assert all(dj_decide(random_balanced_oracle(n, seed=s), n) == "dengeli" for s in range(10))
print("Alıştırma 4 ✓")

### Alıştırma 5 · Rastgele dengeli doğruluk tablosu
`random_balanced_table(n, seed)`: tam 2ⁿ⁻¹ tane 1 içeren rastgele bir doğruluk tablosu (liste) döndürsün. Sonra `oracle_from_truth_table` ile oracle kurup DJ'nin "dengeli" dediğini doğrulayın.

In [ ]:
def random_balanced_table(n, seed=None):
    # TODO
    pass

for s in range(5):
    tt = random_balanced_table(3, seed=s)
    assert len(tt) == 8 and sum(tt) == 4 and set(tt) <= {0, 1}
    assert dj_decide(oracle_from_truth_table(tt), 3) == "dengeli"
assert random_balanced_table(3, seed=1) == random_balanced_table(3, seed=1)     # tekrarlanabilir
print("Alıştırma 5 ✓")

### Alıştırma 6 · Bernstein-Vazirani çözücü
`bv_solve(oracle, n)`: BV devresini tek shot çalıştırıp gizli stringi döndürsün. 8 bitlik 5 rastgele s ile test edilir.

In [ ]:
def bv_solve(oracle, n):
    # TODO
    pass

for _ in range(5):
    s = "".join(rng.choice(["0", "1"], size=8))
    assert bv_solve(bv_oracle(s), 8) == s, s
print("Alıştırma 6 ✓")

### Alıştırma 7 · Klasik BV ve sorgu sayacı
`classical_bv(f)`: `CountingOracle` nesnesini **tam n kez** sorgulayarak s stringini bulsun.

In [ ]:
def classical_bv(f):
    # TODO
    pass

s = "10110011"; n = len(s)
f = CountingOracle([dot2(x, int(s, 2)) for x in range(2**n)])
assert classical_bv(f) == s and f.queries == n
print("Alıştırma 7 ✓  klasik sorgu =", f.queries, " kuantum sorgu = 1")

### Alıştırma 8 · XOR maskeli dengeli oracle
`masked_parity_oracle(s, mask)`: f(x) = s·(x ⊕ mask) mod 2 hesaplayan oracle'ı **X(maske) → CNOT'lar → X(maske)** kalıbıyla kurun (s ve mask string). Doğruluk tablosunu formülle karşılaştırın; oracle'ın devreyi temiz bıraktığını (girdi kübitleri geri döner) kontrol edin.

In [ ]:
def masked_parity_oracle(s, mask):
    # TODO
    pass

s, mask = "0111", "1010"; n = len(s)
o = masked_parity_oracle(s, mask)
expected = [dot2(x ^ int(mask, 2), int(s, 2)) for x in range(2**n)]
assert truth_table_of(o, n) == expected and sum(expected) == 2**(n - 1)
assert dj_decide(o, n) == "dengeli"
# Maske etkisi: BV ölçümü yine s'yi verir (maske yalnızca global işaret ekler)
assert bv_solve(o, n) == s
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- **Oracle** = kara kutu fonksiyon; maliyet = **sorgu sayısı**
- Bit oracle `|x⟩|y⟩ → |x⟩|y⊕f(x)⟩` tersinirdir; hedef **|−⟩** iken **faz oracle** `(−1)^f(x)|x⟩` olur (faz geri tepmesi)
- **H⊗n**: |0…0⟩ → eşit süperpozisyon; |x⟩ → işaretler (−1)^(x·z)
- **Deutsch / DJ**: tek sorguda sabit/dengeli; ölçüm 0…0 ⇔ sabit. Klasik deterministik: 2ⁿ⁻¹+1 sorgu
- **Olasılıksal klasik** algoritma k sorguyla ≤ 2^(1−k) hata → DJ'nin avantajı "kesinlik" avantajıdır
- **BV**: f(x) = s·x, tek sorguda s (klasik n); oracle = s'nin 1 bitlerinden CNOT
- **Simon**: olasılıksal klasiğe karşı bile üstel ayrım

**Gelecek hafta:** Grover arama algoritması — bu haftaki faz oracle'ını "işaretleyici" olarak kullanıp, bir **genlik yükseltme** döngüsüyle aranan elemanı yaklaşık √N sorguda bulacağız.